# 내 실습 환경 확인 (Colab · VS Code 공통)
VS Code에서는 오른쪽 위 **커널 선택 → Python**을 고른 뒤 실행하세요. 빠진 패키지는 아래 설치 칸으로 한 번에 설치해요.

In [1]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [2]:
import importlib, shutil
in_venv = sys.prefix != sys.base_prefix
print('Python', sys.version.split()[0], '|', '가상환경 사용 중 ✅' if in_venv or IN_COLAB else '가상환경 아님 ⚠️ (README의 가상환경 만들기 참고)')
print('실행 중인 파이썬:', sys.executable)
if sys.version_info[:2] != (3, 12) and not IN_COLAB:
    print('⚠️ 수업 기준은 Python 3.12 — README대로 3.12 가상환경(.venv)을 만들고 커널을 바꾸세요')
for pkg in ['pandas', 'matplotlib', 'seaborn', 'openpyxl', 'requests', 'bs4', 'openai', 'dotenv']:
    try:
        m = importlib.import_module(pkg)
        print(f'✅ {pkg:11s}', getattr(m, '__version__', ''))
    except ImportError:
        print(f'❌ {pkg:11s} 설치 필요')

Python 3.12.10 | 가상환경 사용 중 ✅
실행 중인 파이썬: c:\Users\EZ\Downloads\수업코드_데이터저널리즘_3\수업코드_데이터저널리즘\.venv\Scripts\python.exe
✅ pandas      3.1.1
✅ matplotlib  3.11.2
✅ seaborn     0.13.2
✅ openpyxl    3.1.5
✅ requests    2.34.2
✅ bs4         4.15.0
✅ openai      3.48.0
✅ dotenv      


In [3]:
# 빠진 패키지 설치 (Colab에는 대부분 이미 있어요)
%pip install -q pandas matplotlib seaborn openpyxl requests beautifulsoup4 openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## GPU 확인 — 파인튜닝·Ollama에서 씀 (나머지 실습은 CPU로 충분)

In [4]:
try:
    import torch
    print('PyTorch', torch.__version__, '| CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0),
              f'| 메모리 {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')
except ImportError:
    print('PyTorch 없음 — Unsloth Studio·Ollama만 쓸 거면 설치하지 않아도 돼요')

PyTorch 없음 — Unsloth Studio·Ollama만 쓸 거면 설치하지 않아도 돼요


## 내 컴퓨터 도구 확인 (VS Code에서만)

In [5]:
def ver(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=10, shell=True).stdout.strip() or '없음'
    except Exception:
        return '없음'
print('Node.js (n8n은 v20.19~v24):', ver('node -v'))
print('Ollama:', ver('ollama --version'))
import requests
for name, url in [('Ollama 서버', 'http://localhost:11434'), ('Unsloth Studio', 'http://127.0.0.1:8888'), ('n8n', 'http://localhost:5678')]:
    try:
        requests.get(url, timeout=2); print(f'✅ {name} 켜져 있음 ({url})')
    except Exception:
        print(f'⚪ {name} 꺼져 있음 ({url}) — 쓸 때만 켜면 돼요')

Node.js (n8n은 v20.19~v24): v24.21.0
Ollama: 없음
⚪ Ollama 서버 꺼져 있음 (http://localhost:11434) — 쓸 때만 켜면 돼요
⚪ Unsloth Studio 꺼져 있음 (http://127.0.0.1:8888) — 쓸 때만 켜면 돼요
⚪ n8n 꺼져 있음 (http://localhost:5678) — 쓸 때만 켜면 돼요
